# PANDA — Week 2 ensemble OOF evaluation

This notebook measures a fair fold-wise ensemble. For each validation fold,
it only averages models that were trained without that fold's slides, then
reports per-family and ensemble QWK.

**Attach on Kaggle:**
1. `panda-resized-train-data-512x512` (xhlulu)
2. One Kaggle Dataset per model family listed in `MODEL_FAMILIES`

**Settings:** GPU T4 ON, Internet ON


In [ ]:
REPO = 'https://github.com/Shashaboii/AIMI_Panda_Challenge.git'
BRANCH = 'main'
BATCH_SIZE = 16
ORDINAL_MODE = 'threshold'  # threshold or expected for ordinal models
MODEL_FAMILIES = [
    {
        'name': 'b0_smoothl1',
        'weights_dir': '/kaggle/input/panda-effnetb0-5fold-baseline',
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_fold{fold}.pth',
        'model_kind': 'baseline',
    },
    {
        'name': 'b0_ordinal',
        'weights_dir': '/kaggle/input/panda-effnetb0-ordinal-5fold',
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_ordinal_fold{fold}.pth',
        'model_kind': 'baseline',
    },
]


In [ ]:
import os
import subprocess

if os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, '/kaggle/working/repo'], check=True)
subprocess.run(['git', '-C', '/kaggle/working/repo', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
!pip install -q efficientnet_pytorch


In [ ]:
import glob

candidates = (
    glob.glob('/kaggle/input/*resized*512*/train_images/train_images')
    + glob.glob('/kaggle/input/*resized*512*/train_images')
    + glob.glob('/kaggle/input/**/train_images', recursive=True)
)
IMAGE_DIR = next((c for c in candidates if os.path.isdir(c) and len(os.listdir(c)) > 5000), None)
if IMAGE_DIR is None:
    raise RuntimeError(f'Could not find image dir. Input contents: {os.listdir("/kaggle/input")}')
print('IMAGE_DIR:', IMAGE_DIR, 'files:', len(os.listdir(IMAGE_DIR)))


In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, '/kaggle/working/repo')

from src.dataset import PandaDataset
from src.eval import confusion_matrix_str, mean_std_str, qwk
from src.inference import load_model, predict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

df = pd.read_csv('/kaggle/working/repo/data/train_folds.csv')
have = set(os.listdir(IMAGE_DIR))
df = df[df.image_id.apply(lambda x: f'{x}.png' in have)].reset_index(drop=True)
print('Usable slides:', len(df))

for family in MODEL_FAMILIES:
    missing = [
        fold for fold in sorted(df.fold.unique())
        if not os.path.exists(os.path.join(family['weights_dir'], family['weight_pattern'].format(fold=fold)))
    ]
    if missing:
        raise FileNotFoundError(f"{family['name']} is missing folds {missing} in {family['weights_dir']}")

pred_columns = {family['name']: f"pred_{family['name']}" for family in MODEL_FAMILIES}


In [ ]:
family_scores = {family['name']: [] for family in MODEL_FAMILIES}
family_scores['ensemble'] = []
oof_rows = []

for fold in sorted(df.fold.unique()):
    val_df = df[df.fold == fold].reset_index(drop=True)
    dataset = PandaDataset(val_df, IMAGE_DIR, train=False)
    fold_preds = {}

    for family in MODEL_FAMILIES:
        weight_path = os.path.join(
            family['weights_dir'],
            family['weight_pattern'].format(fold=fold),
        )
        model = load_model(
            weight_path,
            backbone=family.get('backbone', 'efficientnet-b0'),
            device=device,
            model_kind=family.get('model_kind', 'baseline'),
        )
        preds = predict(
            model,
            dataset,
            device,
            batch_size=BATCH_SIZE,
            ordinal_mode=ORDINAL_MODE,
        )
        fold_preds[family['name']] = preds
        fold_qwk = qwk(preds, val_df.isup_grade.values)
        family_scores[family['name']].append(fold_qwk)
        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    ensemble_preds = np.mean(np.column_stack(list(fold_preds.values())), axis=1)
    ensemble_qwk = qwk(ensemble_preds, val_df.isup_grade.values)
    family_scores['ensemble'].append(ensemble_qwk)

    print(f'fold {fold}:')
    for family_name, preds in fold_preds.items():
        print(f"  {family_name}: {qwk(preds, val_df.isup_grade.values):.4f}")
    print(f'  ensemble: {ensemble_qwk:.4f}')

    for idx, row in val_df.iterrows():
        out_row = {
            'image_id': row.image_id,
            'fold': int(fold),
            'isup_grade': int(row.isup_grade),
            'pred_ensemble': float(ensemble_preds[idx]),
        }
        for family_name, preds in fold_preds.items():
            out_row[pred_columns[family_name]] = float(preds[idx])
        oof_rows.append(out_row)

oof = pd.DataFrame(oof_rows)
OUTPUT_CSV = '/kaggle/working/ensemble_oof_predictions.csv'
oof.to_csv(OUTPUT_CSV, index=False)
print('Wrote', OUTPUT_CSV)


In [ ]:
summary_rows = []
for family in MODEL_FAMILIES:
    name = family['name']
    pred_col = pred_columns[name]
    summary_rows.append({
        'model': name,
        'mean_std_qwk': mean_std_str(family_scores[name]),
        'global_oof_qwk': qwk(oof[pred_col].values, oof.isup_grade.values),
    })

summary_rows.append({
    'model': 'ensemble',
    'mean_std_qwk': mean_std_str(family_scores['ensemble']),
    'global_oof_qwk': qwk(oof.pred_ensemble.values, oof.isup_grade.values),
})

summary = pd.DataFrame(summary_rows)
display(summary)
print()
print('Ensemble confusion matrix (rows = true, cols = predicted):')
print(confusion_matrix_str(oof.pred_ensemble.values, oof.isup_grade.values))


In [ ]:
for f in sorted(os.listdir('/kaggle/working')):
    p = f'/kaggle/working/{f}'
    if os.path.isfile(p):
        print(f'{f}  ({os.path.getsize(p) / 1e6:.2f} MB)')
